# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyanshu-Technologies/flyrank-ML-track/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The action playbook uses the validated ranking output to prioritize pages for human review.

The primary action is `refresh_review` for pages that receive a high model score and therefore appear near the top of the review queue.

The reason code explains why a page was prioritized. The model score is treated as a directional prioritization signal, not as proof that a refresh will improve performance.

The ranked queue is intended to help a content team allocate limited review capacity. The final action remains with a human reviewer who can consider search intent, strategic importance, content quality, and other information not represented in the model features.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the starter data
df = pd.read_csv(
    "../../data/raw/content_refresh_anonymized.csv"
)

# Observed label
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


# Recreate the Week-5 model features
feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]



# Train model on the full starter
# dataset for playbook generation
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(
    df[feature_columns],
    df["is_declining_label"]
)



# Generate ranking score
df["model_score"] = model.predict_proba(
    df[feature_columns]
)[:, 1]



# Create reason codes
df["reason_code"] = np.select(
    [
        (
            (df["days_since_last_update"] >= 180)
            & (df["impressions_90d"] >= 500)
        ),
        (
            (df["impressions_90d"] >= 500)
            & (df["avg_position"] > 0)
            & (df["avg_position"] <= 10)
        )
    ],
    [
        "stale_visible_page",
        "high_visibility_page"
    ],
    default="model_priority"
)


# Create action labels
df["action"] = np.where(
    df["model_score"] >= df["model_score"].quantile(0.80),
    "refresh_review",
    "monitor"
)



# Build ranked queue
action_queue = df[
    [
        "content_id",
        "client_id",
        "model_score",
        "action",
        "reason_code",
        "days_since_last_update",
        "impressions_90d",
        "avg_position"
    ]
].copy()

action_queue = action_queue.sort_values(
    by="model_score",
    ascending=False
).reset_index(drop=True)

action_queue["rank"] = (
    action_queue.index + 1
)

action_queue = action_queue[
    [
        "rank",
        "content_id",
        "client_id",
        "model_score",
        "action",
        "reason_code",
        "days_since_last_update",
        "impressions_90d",
        "avg_position"
    ]
]


# Display top 20
print("Ranked action queue created.")
print("Rows:", len(action_queue))

display(
    action_queue.head(20)
)


Ranked action queue created.
Rows: 30000


,rank,content_id,client_id,model_score,action,reason_code,days_since_last_update,impressions_90d,avg_position
0,1,content_7ec1abc04dec,client_19581e27de,1.0,refresh_review,high_visibility_page,104,80957,4.8
1,2,content_5882b06e7320,client_7f2253d7e2,1.0,refresh_review,high_visibility_page,20,49382,5.1
2,3,content_d75823fc91dc,client_7f2253d7e2,1.0,refresh_review,model_priority,20,37675,30.5
3,4,content_8b1eeb87a7c5,client_6208ef0f77,1.0,refresh_review,model_priority,104,26142,35.0
4,5,content_a6bc67ad6378,client_19581e27de,1.0,refresh_review,high_visibility_page,104,21671,5.5
5,6,content_5cb5eb3ed482,client_19581e27de,1.0,refresh_review,high_visibility_page,20,23590,5.4
6,7,content_5fe46e04994d,client_4e07408562,1.0,refresh_review,high_visibility_page,104,517715,4.2
7,8,content_e752a4e03dd3,client_6208ef0f77,1.0,refresh_review,model_priority,104,130892,23.9
8,9,content_db0bf34f871c,client_7f2253d7e2,1.0,refresh_review,model_priority,20,22434,40.5
9,10,content_c16ed807a773,client_19581e27de,1.0,refresh_review,high_visibility_page,20,16565,4.5


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

The action queue is intended for content and SEO teams to prioritize which pages should receive human review first.

A high model score means that the page is ranked highly by the learned model based on the observable features used during modeling. The score is directional and should be used to allocate review attention rather than to make an automatic content decision.

The reviewer can use the queue to investigate whether a page should be:
- refreshed,
- expanded,
- protected,
- monitored,
- or considered for another content action.

### Limits

The model does not establish that a page will decline in the future.

It also does not establish that refreshing a page will cause its performance to improve.

The current target is based on an observed declining label rather than a genuinely future outcome. The ranking therefore provides measured decision-support for the current dataset, not a causal or guaranteed prediction.

The model also does not capture every factor that may matter to a content decision, including editorial judgment, strategic importance, search-intent changes, business priorities, and information outside the modeled dataset.

The queue should therefore be treated as a review-prioritization tool and not as an autonomous production system.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*


### Human review rules

Every page selected by the action queue must be reviewed by a human before any content change is made.

The reviewer should check:

1. **Search intent** — Does the page still match the intent behind the queries bringing impressions?
2. **Content quality** — Is the information accurate, useful, complete, and sufficiently current?
3. **Freshness** — Is the page genuinely outdated, or is its age appropriate for the topic?
4. **Search visibility** — Does the page have meaningful observed impressions or other evidence that makes review worthwhile?
5. **Strategic importance** — Does the page matter to the client's content strategy or business goals?
6. **Existing performance** — Is the page already performing well enough that changing it could create unnecessary risk?

The model score is therefore a prioritization signal. The reviewer makes the final decision.

### No-go list

The following decisions should NOT be automated:

- Automatically publishing a content refresh.
- Automatically deleting or pruning a page.
- Automatically changing search-intent targeting.
- Automatically changing titles, headings, or page content.
- Automatically redirecting or canonicalizing a page.
- Automatically deciding that a page has lost search relevance.
- Automatically claiming that a refresh will improve traffic, rankings, clicks, or conversions.
- Automatically overriding editorial or strategic judgment.

The model should not make irreversible or externally visible content changes without human review.

### Action mapping

| Signal / situation | Suggested action | Human check |
|---|---|---|
| High model score | Refresh review | Confirm the page has a genuine content opportunity |
| Stale + visible page | Refresh or expansion review | Check whether the content is actually outdated |
| Strong existing visibility | Protect / monitor review | Avoid unnecessary changes to a successful page |
| Low-confidence or unusual case | Manual investigation | Gather additional context before acting |
| No meaningful signal | Monitor | No immediate content change |


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The action playbook should be monitored periodically rather than treated as permanently valid.

### Monitoring checks

The following checks should be reviewed regularly:

- **Feature availability:** confirm that the inputs required by the model are still available and have not developed unexpected missingness.
- **Feature distribution:** check whether important inputs such as impressions, sessions, content age, and position have shifted substantially.
- **Score distribution:** check whether the distribution of model scores changes substantially from the development data.
- **Action distribution:** check whether the proportion of pages assigned to each action changes unexpectedly.
- **Observed performance:** when a later outcome period becomes available, measure Precision@50 and compare it with the previously measured result.
- **Review quality:** periodically inspect a sample of highly ranked pages to verify that the queue remains useful for human review.

### Retrain triggers

The model should be considered for retraining when:

1. A later evaluation period shows a meaningful deterioration in Precision@50.
2. The distribution of important input features changes substantially.
3. Important model inputs become unavailable or their missingness changes materially.
4. The content environment or measurement process changes enough that the historical training data is no longer representative.
5. A new future-outcome label becomes available that better represents the actual content decision.

Retraining should not happen simply because a more complex model is available. There should be evidence that the current model no longer provides useful decision-support.

### Operational principle

Monitoring is intended to detect when the measured usefulness of the ranking changes. It does not guarantee that the model remains valid indefinitely.

Any retrained model should be evaluated against the existing baseline and the previous model using the same clearly defined evaluation metric and an honest validation design.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
# Exports for the paper
from pathlib import Path

# Create output directories
output_dir = Path("../../work/outputs")
figures_dir = Path("../../work/figures")

output_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)


# 1. Export ranked action queue
queue_path = (
    output_dir / "w07_action_queue.csv"
)

action_queue.to_csv(
    queue_path,
    index=False
)

print("Action queue exported:")
print(queue_path)



# 2. Export model score summary
score_summary = pd.DataFrame({
    "metric": [
        "total_pages",
        "refresh_review_pages",
        "monitor_pages",
        "refresh_review_share"
    ],
    "value": [
        len(action_queue),
        (action_queue["action"] == "refresh_review").sum(),
        (action_queue["action"] == "monitor").sum(),
        (
            action_queue["action"] == "refresh_review"
        ).mean()
    ]
})

metrics_path = (
    output_dir / "w07_playbook_metrics.csv"
)

score_summary.to_csv(
    metrics_path,
    index=False
)

print("\nMetrics exported:")
print(metrics_path)


# 3. Show the exported queue
print("\nTop 20 ranked actions:")

display(
    action_queue.head(20)
)


# 4. Confirm files exist
print("\nExport check:")

print(
    "Queue exists:",
    queue_path.exists()
)

print(
    "Metrics exists:",
    metrics_path.exists()
)

assert queue_path.exists()
assert metrics_path.exists()

print("\nExports: PASSED")


Action queue exported:
../../work/outputs/w07_action_queue.csv

Metrics exported:
../../work/outputs/w07_playbook_metrics.csv

Top 20 ranked actions:


,rank,content_id,client_id,model_score,action,reason_code,days_since_last_update,impressions_90d,avg_position
0,1,content_7ec1abc04dec,client_19581e27de,1.0,refresh_review,high_visibility_page,104,80957,4.8
1,2,content_5882b06e7320,client_7f2253d7e2,1.0,refresh_review,high_visibility_page,20,49382,5.1
2,3,content_d75823fc91dc,client_7f2253d7e2,1.0,refresh_review,model_priority,20,37675,30.5
3,4,content_8b1eeb87a7c5,client_6208ef0f77,1.0,refresh_review,model_priority,104,26142,35.0
4,5,content_a6bc67ad6378,client_19581e27de,1.0,refresh_review,high_visibility_page,104,21671,5.5
5,6,content_5cb5eb3ed482,client_19581e27de,1.0,refresh_review,high_visibility_page,20,23590,5.4
6,7,content_5fe46e04994d,client_4e07408562,1.0,refresh_review,high_visibility_page,104,517715,4.2
7,8,content_e752a4e03dd3,client_6208ef0f77,1.0,refresh_review,model_priority,104,130892,23.9
8,9,content_db0bf34f871c,client_7f2253d7e2,1.0,refresh_review,model_priority,20,22434,40.5
9,10,content_c16ed807a773,client_19581e27de,1.0,refresh_review,high_visibility_page,20,16565,4.5



Export check:
Queue exists: True
Metrics exists: True

Exports: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.